# GenPRM Cross-Verification of Narrative Puzzle Adaptations

This notebook evaluates an **off-the-shelf open-source generative process reward model (GenPRM)** as an independent cross-check of 150 human-audited narrative puzzle adaptations.

## Experimental question
> Given the original puzzle, adapted puzzle, and existing judge verification trace, can GenPRM independently distinguish human-audited **equivalent** from **not-equivalent** adaptations?

## Design
- Fixed evaluation set: **150 audited adaptations** (100 equivalent, 50 not-equivalent).
- GenPRM sees only the original puzzle, final adapted puzzle, and final judge reasoning trace.
- Human/audit labels and judge final verdicts are withheld from the model.
- Primary analysis: GenPRM reward vs human-audited outcome using AUROC/AUPRC and bootstrap CIs.
- Secondary analysis: thresholded accuracy/F1, subgroup diagnostics, and disagreement inspection.

GenPRM was trained primarily for mathematical process supervision, so this is an **out-of-domain independent diagnostic**, not a replacement for human audit.


## 1. Colab setup

Use a GPU runtime. Start with `GenPRM/GenPRM-1.5B`. The official repository exposes `GenPRM.inference(...)`, returning generated verification reasoning and a reward.


In [ ]:
# Official GenPRM implementation
!git clone -q https://github.com/RyanLiu112/GenPRM.git
%cd /content/GenPRM/src
!pip install -q -r requirements.txt
!pip install -q pandas numpy scikit-learn matplotlib tqdm


In [ ]:
import os, re, json, random, warnings, subprocess, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "GenPRM/GenPRM-1.5B"
OUTPUT_DIR = Path("/content/genprm_adaptation_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / "genprm_predictions.csv"


## 2. Load the 150-case audit dataset

Upload `Sheet1 copy-Grid view.csv` when prompted.

The supplied file has `corrected_outcome` populated for all 150 cases. The notebook prints its relationship to `final_outcome` and `human_decision` before defining the target, so the annotation mapping is explicit.


In [ ]:
from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded.keys()))
print("Using:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head(2))


In [ ]:
required = [
    "Puzzle_Number", "Question_Number", "original_sample",
    "final_adapted_puzzle", "final_judge_think_trace",
    "final_outcome", "corrected_outcome"
]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
assert len(df) == 150, f"Expected 150 rows, found {len(df)}"

print("Final outcome:")
display(df["final_outcome"].value_counts(dropna=False).to_frame("n"))

print("Corrected/audited outcome:")
display(df["corrected_outcome"].value_counts(dropna=False).to_frame("n"))

if "human_decision" in df.columns:
    print("Explicit human_decision field:")
    display(df["human_decision"].value_counts(dropna=False).to_frame("n"))

print("final_outcome × corrected_outcome:")
display(pd.crosstab(df["final_outcome"], df["corrected_outcome"], dropna=False))

LABEL_MAP = {"equivalent": 1, "not_equivalent": 0}
df["human_gold"] = (
    df["corrected_outcome"].astype(str).str.strip().str.lower().map(LABEL_MAP)
)
assert df["human_gold"].notna().all()

print("Evaluation target:")
display(
    df["human_gold"].value_counts().sort_index()
      .rename(index={0:"not_equivalent", 1:"equivalent"}).to_frame("n")
)


## 3. Leakage-safe GenPRM input

GenPRM receives only:
1. original puzzle,
2. adapted puzzle,
3. existing judge reasoning trace.

It does **not** receive `final_judge_verdict`, `final_equivalence`, `final_outcome`, `corrected_outcome`, `human_decision`, `audit_notes`, or other audit outputs.

The complete judge trace is treated as **one process unit**. This avoids adding an arbitrary trace-segmentation method under deadline.


In [ ]:
SYSTEM_PROMPT = """You are an independent process verifier.

Your task is to review a reasoning trace that evaluates whether a narrative adaptation of a logical puzzle preserves the original puzzle's logical structure.

Evaluate the verification reasoning using only the original puzzle, adapted puzzle, and reasoning trace supplied to you.

Relevant preservation properties include:
- one-to-one entity correspondence and consistent entity use;
- ordering relations;
- adjacency and distance relations;
- absolute-position constraints;
- conditional dependencies;
- disjunctive constraints;
- grouping constraints;
- counting/cardinality constraints;
- preservation of question semantics;
- preservation of answer-choice semantics;
- missing, added, merged, split, or altered constraints.

Narrative wording and entity names may change.
Do not use any external verdict or human label.
Review whether the supplied verification reasoning is logically supported by the original and adapted puzzles.
"""

def clean_text(x):
    return "" if pd.isna(x) else str(x).strip()

def build_messages(row):
    user = f"""ORIGINAL PUZZLE:
{clean_text(row['original_sample'])}

ADAPTED PUZZLE:
{clean_text(row['final_adapted_puzzle'])}

VERIFICATION REASONING TRACE:
{clean_text(row['final_judge_think_trace'])}

Review the verification reasoning above as a single process unit."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]

example = build_messages(df.iloc[0])
print(example[0]["content"])
print("\n--- USER INPUT PREVIEW ---\n")
print(example[1]["content"][:5000])


## 4. Load GenPRM

This uses the official repository wrapper instead of inventing a custom score from generated text.


In [ ]:
from prm_evaluation.genprm_inference import GenPRM, CodeExecutor

genprm = GenPRM(MODEL_NAME)
code_executor = CodeExecutor()
print("Loaded:", MODEL_NAME)


## 5. Smoke test

Run two examples before launching all 150. Confirm that a numeric reward is produced.


In [ ]:
def reward_to_float(reward):
    # Convert common scalar/list/tensor reward formats to one float.
    if reward is None:
        return np.nan
    if hasattr(reward, "detach"):
        arr = reward.detach().float().cpu().numpy().reshape(-1)
        return float(arr[-1])
    if isinstance(reward, np.ndarray):
        return float(reward.reshape(-1)[-1])
    if isinstance(reward, (list, tuple)):
        return np.nan if len(reward) == 0 else reward_to_float(reward[-1])
    try:
        return float(reward)
    except Exception:
        nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", str(reward))
        return float(nums[-1]) if nums else np.nan

def run_one(row):
    messages = build_messages(row)
    output, reward = genprm.inference(
        messages, cur_step=1, code_executor=code_executor
    )
    reasoning = output[0] if isinstance(output, (list, tuple)) and output else str(output)
    return {
        "Puzzle_Number": row["Puzzle_Number"],
        "Question_Number": row["Question_Number"],
        "genprm_reasoning": reasoning,
        "genprm_reward_raw": str(reward),
        "genprm_reward": reward_to_float(reward),
    }

smoke = []
for _, row in df.head(2).iterrows():
    result = run_one(row)
    smoke.append(result)
    print("=" * 80)
    print("ID:", result["Puzzle_Number"], result["Question_Number"])
    print("Reward:", result["genprm_reward"])
    print("Reasoning preview:", result["genprm_reasoning"][:1500])

display(pd.DataFrame(smoke)[["Puzzle_Number","Question_Number","genprm_reward"]])


## 6. Run all 150 with checkpointing

The loop saves after every example and skips completed puzzle/question IDs when resumed.


In [ ]:
# For persistent Colab storage, optionally mount Drive and redefine OUTPUT_DIR/RESULTS_CSV.
id_cols = ["Puzzle_Number", "Question_Number"]

if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    completed = set(zip(results_df["Puzzle_Number"], results_df["Question_Number"]))
    records = results_df.to_dict("records")
    print(f"Resuming with {len(completed)} completed cases.")
else:
    completed, records = set(), []

errors = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    key = (row["Puzzle_Number"], row["Question_Number"])
    if key in completed:
        continue
    try:
        result = run_one(row)
        result["error"] = ""
    except Exception as e:
        result = {
            "Puzzle_Number": row["Puzzle_Number"],
            "Question_Number": row["Question_Number"],
            "genprm_reasoning": "",
            "genprm_reward_raw": "",
            "genprm_reward": np.nan,
            "error": repr(e),
        }
        errors.append((key, repr(e)))
    records.append(result)
    pd.DataFrame(records).to_csv(RESULTS_CSV, index=False)

print("Saved:", RESULTS_CSV)
print("Errors:", len(errors))


## 7. Merge predictions with human audit labels

Labels are merged **after inference**, ensuring they never enter the model prompt.


In [ ]:
pred = pd.read_csv(RESULTS_CSV)
eval_df = df.merge(
    pred, on=["Puzzle_Number", "Question_Number"],
    how="left", validate="one_to_one"
)

print("Rows:", len(eval_df))
print("Missing GenPRM rewards:", eval_df["genprm_reward"].isna().sum())

display(eval_df[
    ["Puzzle_Number","Question_Number","corrected_outcome",
     "human_gold","genprm_reward","error"]
].head())

eval_df.to_csv(OUTPUT_DIR / "genprm_merged_evaluation.csv", index=False)


## 8. Primary metrics

Because GenPRM is out-of-domain and its score is not assumed calibrated for this task, **AUROC** is the primary metric. AUPRC is also reported.

The cell checks score direction. Any inversion is recorded explicitly rather than hidden.


In [ ]:
valid = eval_df.dropna(subset=["genprm_reward", "human_gold"]).copy()

raw_auc = roc_auc_score(valid["human_gold"], valid["genprm_reward"])
if raw_auc >= 0.5:
    valid["score_equivalent"] = valid["genprm_reward"]
    SCORE_DIRECTION = "higher reward = more equivalent"
else:
    valid["score_equivalent"] = -valid["genprm_reward"]
    SCORE_DIRECTION = "lower reward = more equivalent; inverted for discrimination metrics"

auc = roc_auc_score(valid["human_gold"], valid["score_equivalent"])
auprc = average_precision_score(valid["human_gold"], valid["score_equivalent"])

print("N =", len(valid))
print("Raw AUROC =", round(raw_auc, 4))
print("Score direction:", SCORE_DIRECTION)
print(f"AUROC: {auc:.4f}")
print(f"AUPRC: {auprc:.4f}")


## 9. Stratified bootstrap 95% confidence intervals


In [ ]:
def stratified_bootstrap_metric(data, metric_fn, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    pos = data[data["human_gold"] == 1]
    neg = data[data["human_gold"] == 0]
    vals = []
    for _ in range(n_boot):
        p = pos.iloc[rng.integers(0, len(pos), len(pos))]
        n = neg.iloc[rng.integers(0, len(neg), len(neg))]
        b = pd.concat([p, n], ignore_index=True)
        vals.append(metric_fn(b["human_gold"], b["score_equivalent"]))
    return np.percentile(vals, [2.5, 50, 97.5]), np.asarray(vals)

auc_ci, _ = stratified_bootstrap_metric(valid, roc_auc_score)
ap_ci, _ = stratified_bootstrap_metric(valid, average_precision_score)

print(f"AUROC = {auc:.3f} (95% bootstrap CI {auc_ci[0]:.3f}–{auc_ci[2]:.3f})")
print(f"AUPRC = {auprc:.3f} (95% bootstrap CI {ap_ci[0]:.3f}–{ap_ci[2]:.3f})")


## 10. Thresholded secondary metrics

The threshold is selected on this diagnostic set using Youden's J, so thresholded metrics are descriptive. AUROC is the cleaner headline result.


In [ ]:
fpr, tpr, thresholds = roc_curve(valid["human_gold"], valid["score_equivalent"])
best_i = np.nanargmax(tpr - fpr)
threshold = thresholds[best_i]
valid["genprm_pred"] = (valid["score_equivalent"] >= threshold).astype(int)

acc = accuracy_score(valid["human_gold"], valid["genprm_pred"])
bacc = balanced_accuracy_score(valid["human_gold"], valid["genprm_pred"])
precision, recall, f1, _ = precision_recall_fscore_support(
    valid["human_gold"], valid["genprm_pred"], average="binary", zero_division=0
)
cm = confusion_matrix(valid["human_gold"], valid["genprm_pred"])

metrics = pd.DataFrame([{
    "N": len(valid), "raw_AUROC": raw_auc, "AUROC": auc,
    "AUROC_CI_low": auc_ci[0], "AUROC_CI_high": auc_ci[2],
    "AUPRC": auprc, "AUPRC_CI_low": ap_ci[0], "AUPRC_CI_high": ap_ci[2],
    "threshold": threshold, "accuracy": acc, "balanced_accuracy": bacc,
    "precision_equivalent": precision, "recall_equivalent": recall,
    "F1_equivalent": f1, "score_direction": SCORE_DIRECTION,
}])

display(metrics.round(4))
metrics.to_csv(OUTPUT_DIR / "paper_metrics.csv", index=False)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)


## 11. Publication-ready figures


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
groups = [
    valid.loc[valid["human_gold"] == 1, "score_equivalent"].values,
    valid.loc[valid["human_gold"] == 0, "score_equivalent"].values,
]
ax.boxplot(groups, tick_labels=[
    f"Equivalent (n={(valid.human_gold==1).sum()})",
    f"Not equivalent (n={(valid.human_gold==0).sum()})"
])
ax.set_ylabel("GenPRM verification score")
ax.set_title("Independent GenPRM scores by audited outcome")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "genprm_score_distribution.pdf", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "genprm_score_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.0))
fpr, tpr, _ = roc_curve(valid["human_gold"], valid["score_equivalent"])
ax.plot(fpr, tpr, label=f"GenPRM (AUROC={auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("GenPRM cross-verification ROC")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "genprm_roc.pdf", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "genprm_roc.png", dpi=300, bbox_inches="tight")
plt.show()


## 12. Exploratory subgroup diagnostics


In [ ]:
def subgroup_auc(data, column, min_n=10):
    rows = []
    for value, g in data.groupby(column, dropna=False):
        if len(g) < min_n or g["human_gold"].nunique() < 2:
            continue
        rows.append({
            "group": value, "N": len(g),
            "positive_n": int(g["human_gold"].sum()),
            "negative_n": int((1-g["human_gold"]).sum()),
            "AUROC": roc_auc_score(g["human_gold"], g["score_equivalent"]),
        })
    return pd.DataFrame(rows).sort_values("N", ascending=False) if rows else pd.DataFrame()

for col in ["generator_model", "judge_model", "iterations_used"]:
    if col in valid.columns:
        print("\n###", col)
        table = subgroup_auc(valid, col)
        display(table)
        table.to_csv(OUTPUT_DIR / f"subgroup_{col}.csv", index=False)


## 13. Inspect GenPRM–human disagreements


In [ ]:
disagreements = valid[valid["genprm_pred"] != valid["human_gold"]].copy()
disagreements["margin"] = abs(disagreements["score_equivalent"] - threshold)
disagreements = disagreements.sort_values("margin", ascending=False)

cols = [
    "Puzzle_Number","Question_Number","corrected_outcome",
    "genprm_reward","genprm_reasoning"
]
print("Disagreements:", len(disagreements))
display(disagreements[cols].head(20))
disagreements[cols].to_csv(
    OUTPUT_DIR / "genprm_human_disagreements.csv", index=False
)


## 14. Optional robustness

If time permits, rerun each case 3 times and average the reward to measure stochastic stability. Under a deadline, keep the one-pass 150-case experiment as the primary analysis.


In [ ]:
RUN_REPEATS = False
N_REPEATS = 3
# Optional extension intentionally disabled.


## 15. Paper-ready result template

> **Independent process-verifier cross-check.** To complement the LLM-based structural verification and human audit, we evaluated an off-the-shelf open-source generative process reward model, GenPRM-1.5B, on the same stratified set of 150 audited narrative adaptations (100 equivalent and 50 non-equivalent). GenPRM received the original puzzle, adapted puzzle, and the primary judge's reasoning trace, while final judge verdicts and human/audit labels were withheld. We treated the complete verification trace as a single process unit and used the model's verification reward as an independent confidence score. Despite GenPRM being trained primarily for mathematical process supervision, its scores achieved an AUROC of **[AUROC]** (95% bootstrap CI **[LOW–HIGH]**) against the audited equivalence labels. This analysis is intended as an out-of-domain robustness cross-check rather than as an additional source of ground-truth labels.

### Interpretation
- **High AUROC:** useful independent evidence supporting the audited distinction.
- **~0.5 AUROC:** poor cross-domain transfer; do not oversell it.
- **Inverted raw AUROC:** inspect reward semantics and generated critiques; report score orientation transparently.
- Do not describe GenPRM as establishing ground truth.


## 16. Reproducibility metadata


In [ ]:
metadata = {
    "model": MODEL_NAME,
    "seed": SEED,
    "n_examples": len(df),
    "n_equivalent": int((df["human_gold"] == 1).sum()),
    "n_not_equivalent": int((df["human_gold"] == 0).sum()),
    "label_source": "corrected_outcome",
    "trace_granularity": "complete final_judge_think_trace as one process unit",
    "score_direction": SCORE_DIRECTION,
    "python": platform.python_version(),
}
try:
    metadata["genprm_git_commit"] = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip()
except Exception:
    metadata["genprm_git_commit"] = None

with open(OUTPUT_DIR / "run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))
print("\nArtifacts:", OUTPUT_DIR)
